# RLHF on Kaggle GPU — SFT → Reward Model → PPO

Run order: **Settings ▸ Accelerator = GPU (T4 x2 / P100)** and **Internet = On**, then run all.

**Getting the code onto Kaggle** (pick one):
1. Push this repo to GitHub and set `REPO_URL` below, **or**
2. Zip the repo, upload it as a Kaggle *Dataset*, and attach it (it appears under `/kaggle/input/...`).

In [ ]:
import torch, subprocess
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — enable the accelerator!')
# Kaggle ships torch; refresh the post-training libs.
!pip install -q -U 'transformers>=4.44' 'datasets>=2.20' 'accelerate>=0.33' 'peft>=0.12'

In [ ]:
import os
REPO_URL = ''                                   # e.g. 'https://github.com/<you>/RLHF-pipeline.git'
ATTACHED = '/kaggle/input/rlhf-pipeline'         # if you attached the repo as a Dataset
if REPO_URL:
    !rm -rf /kaggle/working/rlhf-pipeline && git clone -q $REPO_URL /kaggle/working/rlhf-pipeline
elif os.path.exists(ATTACHED):
    !rm -rf /kaggle/working/rlhf-pipeline && cp -r $ATTACHED /kaggle/working/rlhf-pipeline
else:
    raise SystemExit('Set REPO_URL or attach the repo as a Kaggle Dataset at ATTACHED.')
%cd /kaggle/working/rlhf-pipeline
!ls

In [ ]:
# Quick correctness check (tiny model, seconds) before the real run.
!python scripts/smoke_test.py && python -m pytest tests/ -q

In [ ]:
# Base model + dataset budget. pythia-160m fits comfortably on a T4; bump up if you have headroom.
MODEL = 'EleutherAI/pythia-160m'   # try 'Qwen/Qwen2.5-0.5B' or 'EleutherAI/pythia-410m'
print('model:', MODEL)

## 1. SFT

In [ ]:
!python scripts/train_sft.py \
  -o model.name_or_path=$MODEL -o data.max_samples=8000 \
  -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/sft

## 2. Reward model

In [ ]:
!python scripts/train_reward_model.py \
  -o model.name_or_path=$MODEL -o data.max_samples=20000 -o data.max_eval_samples=500 \
  -o train.batch_size=8 -o train.bf16=true -o output_dir=checkpoints/reward_model

## 3. PPO (RL against the reward model)

In [ ]:
!python scripts/train_ppo.py \
  -o policy.name_or_path=checkpoints/sft \
  -o reward_model.name_or_path=checkpoints/reward_model \
  -o ppo.total_episodes=4096 -o ppo.rollout_batch_size=32 -o ppo.mini_batch_size=8 \
  -o generation.max_new_tokens=48 -o data.max_prompt_length=160

## 4. Did it help? Win-rate of PPO vs SFT under the reward model

In [ ]:
!python scripts/evaluate.py score-policy --policy checkpoints/ppo \
  --reward-model checkpoints/reward_model --compare checkpoints/sft \
  --num 100 --max-new-tokens 48

### Notes
- For GRPO instead of PPO: swap the cell for `scripts/train_grpo.py` with the same `policy`/`reward_model` overrides.
- Out of memory? add `-o policy.use_lora=true` (PPO) / `-o model.use_lora=true` (SFT/RM), lower `rollout_batch_size`, or shorten `generation.max_new_tokens`.
- Checkpoints + `metrics.jsonl` land under `checkpoints/`; download them from the Kaggle output panel.